# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5, (2,3)  : создается матрица [ 0, 1 , 2
                                   3, 4 , 5] , так как индекс идет с нуля берем элемент с первой строки, и последнего столбца, то есть 5. Размер матрицы (2,3)  
2. 1, 8 : Возметься первое появление максимума, 8ки, поэтому 1ая позиция
3. 99 : мы проходили то что срез, не копия, а ссылка на np.array, поэтому смена элемента повлияет на изначальный список, поэтому ответ 99
4. 12, 6 : Берем только элементы больше 3, это 5 и 7, их сумма 12, а среднее 6
5. (2,3) , 0.2 : У нулевой матрицы все равно размерность 2,3 , а в linspace(0,1,5) , мы создали список от 0 до 1 5 точек, это [0, 0.25, 0.5, 0.75, 1.00] где на 1ом индексе стоит 0.25

In [2]:
import numpy as np
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


In [3]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [6]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])
print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [51]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
clean_unpaid = keep & (cust != "TEST") & ~paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


In [60]:
# База
#
cities = np.unique(city)
amount_clean = amount[clean]
amount_clean_unpaid = amount[clean_unpaid]
max_index = amount_clean.argmax()
cust_clean = cust[clean]
cities_clean = city[clean]
cities_clean_unpaid = city[clean_unpaid]
print("Сумма заказа", amount_clean[max_index], "Клиент", cust_clean[max_index], "Город", cities_clean[max_index])
# 2
paid_sum = amount_clean.sum()
unpaid_sum = amount_clean_unpaid.sum()
print("Выручка оплаченных заказов: " , paid_sum, ". Сумма неоплаченных заказов: ", unpaid_sum)
# 3 (Если учитывать что тут спрашивают сколько всех заказов(оплаченных и неоплаченных) и доля тоже от всех заказов(оплаченных и неоплаченных))
total_without_test = paid_sum + unpaid_sum
total_orders_shymkent = sum(amount_clean[cities_clean == "Шымкент"].shape + amount_clean_unpaid[cities_clean_unpaid == "Шымкент"].shape)
total_sum_shymkent = amount_clean[cities_clean == "Шымкент"].sum() + amount_clean_unpaid[cities_clean_unpaid == "Шымкент"].sum()
print("Заказов в Шымкенте: ", total_orders_shymkent, "Доля:", total_sum_shymkent/total_without_test * 100, "%")

Сумма заказа 139040.0 Клиент C02122 Город Шымкент
Выручка оплаченных заказов:  69662930.0 . Сумма неоплаченных заказов:  7778880.0
Заказов в Шымкенте:  776 Доля: 14.134432033548801 %


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [81]:
# Стандарт
# 4
orders_50k_with_test = amount[amount > 50000 & paid].sum() * 0.15
orders_50k_without_test = amount_clean[amount_clean > 50000].sum() * 0.15
print("С клиентами TEST:", orders_50k_with_test, " ", "Без:" , orders_50k_without_test)
# Потому что у тестовых заказов очень огромные суммы, и если их убрать то соответсвенно упадет потеря выручки"
print(cust[amount > 1000000])

# 5 (уберу только тестовые, оставлю неоплаченные)
clean_paid_unpaid = keep & (cust != "TEST")
days_clean_paid_unpaid = days[clean_paid_unpaid]
cities_clean_paid_unpaid = city[clean_paid_unpaid]
cities_1_2 = cities_clean_paid_unpaid[days_clean_paid_unpaid < 3]
for c in cities:
  print("Доля заказов, доставленных за 1-2 дня для города" , c , "составляет:" , cities_1_2[cities_1_2 == c].shape[0] / cities_clean_paid_unpaid[cities_clean_paid_unpaid == c].shape[0] * 100 , "%")

# 6
avg = amount.mean()
new_avg = amount_clean.mean()
print("Средний чек (Старый):", round(avg))
print("Средний чек (Новый):", round(new_avg))
# проблема в том, что мы не убрали тест, и не сделали выборку paid unpaid, поэтому просто берем и находим по amount_clean



С клиентами TEST: 16482504.0   Без: 1167384.0
['TEST' 'TEST' 'TEST']
Доля заказов, доставленных за 1-2 дня для города Актобе составляет: 62.01550387596899 %
Доля заказов, доставленных за 1-2 дня для города Алматы составляет: 64.60302457466919 %
Доля заказов, доставленных за 1-2 дня для города Астана составляет: 63.85362210604929 %
Доля заказов, доставленных за 1-2 дня для города Атырау составляет: 63.63636363636363 %
Доля заказов, доставленных за 1-2 дня для города Караганда составляет: 67.2447013487476 %
Доля заказов, доставленных за 1-2 дня для города Шымкент составляет: 28.994845360824744 %
Средний чек (Старый): 20114
Средний чек (Новый): 14700


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [94]:
# Челлендж
# 7
import time

x = np.random.rand(1_000_000)

t = time.time()
amount_paid = 0
for r in rows:
  if r["status"] == "оплачен":
    amount_paid += float(r["amount"].replace(" ", ""))
t_loop = time.time() - t

t = time.time()
amount_paid = amount[paid].sum()
t_np = time.time() - t

print(f"цикл: {t_loop:.3f} c, NumPy: {t_np:.4f} c, быстрее в {t_loop / t_np:.0f} раз")
np.random.seed(42)
group_A = np.random.rand(1000)
group_B = np.random.rand(1000)
group_A_purchased = group_A[group_A < 0.05]
group_B_purchased = group_B[group_B < 0.06]

print("Покупок в группе А: " , group_A_purchased.shape[0], "Покупок в группе B:" , group_B_purchased.shape[0])


цикл: 0.002 c, NumPy: 0.0002 c, быстрее в 10 раз
Покупок в группе А:  54 Покупок в группе B: 58


In [96]:
counter = 0
for i in range(100):
  np.random.seed(i)
  group_A = np.random.rand(1000)
  group_B = np.random.rand(1000)
  group_A_purchased = group_A[group_A < 0.05]
  group_B_purchased = group_B[group_B < 0.06]
  print("Покупок в группе А: " , group_A_purchased.shape[0], "Покупок в группе B:" , group_B_purchased.shape[0])
  if group_B_purchased.shape[0] > group_A_purchased.shape[0]:
    counter += 1

print("В стольких случаях В оказалась лучше А: " , counter)

Покупок в группе А:  45 Покупок в группе B: 66
Покупок в группе А:  57 Покупок в группе B: 61
Покупок в группе А:  43 Покупок в группе B: 63
Покупок в группе А:  41 Покупок в группе B: 48
Покупок в группе А:  51 Покупок в группе B: 57
Покупок в группе А:  50 Покупок в группе B: 49
Покупок в группе А:  43 Покупок в группе B: 65
Покупок в группе А:  49 Покупок в группе B: 55
Покупок в группе А:  55 Покупок в группе B: 54
Покупок в группе А:  46 Покупок в группе B: 62
Покупок в группе А:  60 Покупок в группе B: 65
Покупок в группе А:  40 Покупок в группе B: 52
Покупок в группе А:  44 Покупок в группе B: 59
Покупок в группе А:  48 Покупок в группе B: 55
Покупок в группе А:  37 Покупок в группе B: 53
Покупок в группе А:  62 Покупок в группе B: 56
Покупок в группе А:  49 Покупок в группе B: 48
Покупок в группе А:  53 Покупок в группе B: 60
Покупок в группе А:  62 Покупок в группе B: 73
Покупок в группе А:  43 Покупок в группе B: 56
Покупок в группе А:  56 Покупок в группе B: 65
Покупок в гру

В 79 случаях из 100, В оказалась лучше А. Это говорит о том что стабильно в группе В больше заказов, так как там больше вероятность, но это не обязывает ее быть всегда выше, поэтому иногда у А больше заказов. Это говорит о том что размер выборки недостаточно большой, чтобы больше выявлять преимущество В над А.